# **Enron mail dataset에서 업무용 이메일 샘플링**


> 업무 이메일 필터링은 다음 방식으로 넣었습니다.

- 제목/본문에서 업무 관련 키워드 포함 여부 확인

- 너무 짧은 메일 제거

- RE:, FW:, FWD: 로 시작하는 회신/전달 중심 제목

- 회신/전달 흔적만 있고 내용이 거의 없는 메일 일부 제거

- 자동응답/개인성 강한 메일 패턴 일부 제외

***즉, “Enron 전체 메일 중에서 업무 관련 패턴이 포함된 이메일만 우선 선별한 뒤 분석용 샘플 300개를 구성***

**파일 구조**

```
Capstone_AI2/
 └── reference_data/
      ├── enron_business_email_sample_300_cleaned.csv
      ├── enron_subject_patterns.csv
      ├── enron_greeting_patterns.csv
      ├── enron_closing_patterns.csv
      ├── enron_request_patterns.csv  
      └── enron_pattern_summary.csv
```

- **enron_business_email_sample_300_cleaned.csv**: 실제 Enron Email Dataset에서 추출한 업무 이메일 샘플 데이터 (300개)

- **enron_subject_patterns.csv** : Enron 이메일 제목 패턴 분석 결과

- **enron_greeting_patterns.csv** : 이메일 시작 인사 패턴 분석 결과

- **enron_closing_patterns.csv** : 이메일 종료 인사 패턴 분석 결과

- **enron_request_patterns.csv** : 이메일 본문에서 나타나는 업무 요청 표현 패턴 분석 결과

- **enron_pattern_summary.csv** : Enron 이메일 패턴을 업무 유형별로 요약한 데이터

# **Ⅰ. Enroll email 데이터셋 로드 및 분석**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_PATH = "/content/drive/MyDrive/Capstone_AI2"
REF_PATH = os.path.join(BASE_PATH, "reference_data")

os.makedirs(REF_PATH, exist_ok=True)

print("Reference data path:", REF_PATH)

In [ ]:
# ==========================================
# 0. 라이브러리 설치
# ==========================================
!pip install pandas tqdm

import os
import re
import pandas as pd
from tqdm import tqdm
from email import policy
from email.parser import BytesParser

# ==========================================
# 1. Enron Dataset 다운로드 및 압축 해제
# ==========================================
!wget -nc https://www.cs.cmu.edu/~./enron/enron_mail_20110402.tgz
!tar -xzf enron_mail_20110402.tgz

DATA_DIR = "enron_mail_20110402/maildir"
print("DATA_DIR:", DATA_DIR)

# ==========================================
# 2. 이메일 파일 경로 수집
# ==========================================
email_files = []

for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        email_files.append(os.path.join(root, file))

print("Total email files found:", len(email_files))

# ==========================================
# 3. 텍스트 정리 함수
# ==========================================
def clean_text(text):
    if text is None:
        return ""
    text = str(text)
    text = text.replace("\x00", " ")
    text = re.sub(r"\r\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

# ==========================================
# 4. 회신/전달 prefix 제거 함수
# ==========================================
def normalize_subject(subject):
    if not subject:
        return ""
    s = subject.strip()
    s = re.sub(r"^(re|fw|fwd)\s*:\s*", "", s, flags=re.IGNORECASE)
    return s.strip()

# ==========================================
# 5. quoted text / forwarded text 일부 제거
# ==========================================
def remove_quoted_text(body):
    if not body:
        return ""

    lines = body.split("\n")
    cleaned_lines = []

    stop_patterns = [
        r"^-{2,}\s*Original Message\s*-{2,}$",
        r"^-{2,}\s*Forwarded by",
        r"^From:",
        r"^Sent:",
        r"^To:",
        r"^Cc:",
        r"^Subject:",
        r"^>+"
    ]

    for line in lines:
        stripped = line.strip()
        should_stop = False

        for p in stop_patterns:
            if re.search(p, stripped, flags=re.IGNORECASE):
                should_stop = True
                break

        if should_stop:
            break

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines).strip()

# ==========================================
# 6. 이메일 파싱 함수
# ==========================================
def parse_email(file_path):
    try:
        with open(file_path, "rb") as f:
            msg = BytesParser(policy=policy.default).parse(f)

        subject = msg["subject"]
        body = ""

        if msg.is_multipart():
            text_parts = []
            for part in msg.walk():
                content_type = part.get_content_type()
                content_disposition = str(part.get("Content-Disposition", ""))

                if content_type == "text/plain" and "attachment" not in content_disposition.lower():
                    try:
                        part_text = part.get_content()
                        if part_text:
                            text_parts.append(str(part_text))
                    except:
                        pass
            body = "\n".join(text_parts)
        else:
            try:
                body = msg.get_content()
            except:
                body = ""

        subject = clean_text(subject if subject else "")
        body = clean_text(body if body else "")
        body = remove_quoted_text(body)

        # 길이 제한
        body = body[:5000]

        email_text = clean_text(subject + " " + body)

        if len(email_text) < 30:
            return None

        return {
            "subject": subject,
            "normalized_subject": normalize_subject(subject),
            "body": body,
            "email_text": email_text
        }

    except:
        return None

# ==========================================
# 7. 업무 이메일 필터링 함수
# ==========================================
business_keywords = [
    "meeting", "schedule", "appointment", "availability",
    "contract", "agreement", "proposal", "project",
    "quote", "quotation", "pricing", "price", "cost",
    "invoice", "payment", "budget", "purchase",
    "client", "customer", "service", "support",
    "report", "update", "request", "deadline",
    "approval", "document", "file", "attached",
    "confirm", "confirmation", "review", "discussion",
    "conference", "call", "presentation", "issue",
    "problem", "system", "access", "account",
    "application", "interview", "resume", "candidate",
    "vacation", "leave", "notice", "announcement",
    "marketing", "campaign", "media", "press",
    "partnership", "sponsorship", "collaboration",
    "proposal", "quotation", "contract", "report"
]

exclude_keywords = [
    "porn", "sex", "casino", "gambling"
]

noise_patterns = [
    r"out of office",
    r"automatic reply",
    r"auto reply",
    r"delivery status notification",
    r"undeliverable",
    r"returned mail",
    r"mail failure",
    r"do not reply"
]

# 개인성/회신성 subject 제거
personal_subject_patterns = [
    r"^(re|fw|fwd)\s*:",
    r"\bgame\b",
    r"\blunch\b",
    r"\bdinner\b",
    r"\bparty\b",
    r"\bbirthday\b",
    r"\bdrinks\b",
    r"\bgolf\b",
    r"\bmovie\b"
]

def is_business_email(subject, body, email_text):
    text = email_text.lower()
    subj = subject.lower().strip()

    if len(text) < 50:
        return False

    for kw in exclude_keywords:
        if kw in text:
            return False

    for pattern in noise_patterns:
        if re.search(pattern, text):
            return False

    # 제목이 RE:/FW:/FWD: 로 시작하면 제거
    for pattern in personal_subject_patterns:
        if re.search(pattern, subj, flags=re.IGNORECASE):
            return False

    matched_keywords = [kw for kw in business_keywords if kw in text]

    if len(subj.strip()) < 2 and len(matched_keywords) == 0:
        return False

    # 업무 문장 패턴
    business_patterns = [
        r"please let me know",
        r"please review",
        r"please confirm",
        r"attached (is|are)",
        r"i would like to",
        r"could you",
        r"can you",
        r"let.?s discuss",
        r"for your review",
        r"for your approval",
        r"thank you for your email",
        r"please find attached",
        r"we would like to",
        r"request for",
        r"confirm receipt",
        r"schedule a meeting"
    ]

    # 키워드 1개 이상 + 업무패턴 일부 있으면 더 강하게 통과
    if len(matched_keywords) >= 1:
        return True

    for pattern in business_patterns:
        if re.search(pattern, text):
            return True

    return False

# ==========================================
# 8. 이메일 파싱 및 필터링
# ==========================================
parsed_data = []
MAX_PARSE = 50000

for file_path in tqdm(email_files[:MAX_PARSE], desc="Parsing emails"):
    result = parse_email(file_path)
    if result is not None:
        parsed_data.append(result)

df = pd.DataFrame(parsed_data)
print("Parsed emails:", len(df))

business_df = df[
    df.apply(lambda row: is_business_email(row["subject"], row["body"], row["email_text"]), axis=1)
].copy()

print("Business-like emails:", len(business_df))

# ==========================================
# 9. 중복 제거
# ==========================================
business_df.drop_duplicates(subset=["normalized_subject", "body"], inplace=True)
business_df.reset_index(drop=True, inplace=True)

print("After deduplication:", len(business_df))

# ==========================================
# 10. 샘플링
# ==========================================
if len(business_df) < 300:
    raise ValueError(f"업무 이메일이 300개보다 적습니다. 현재 개수: {len(business_df)} / MAX_PARSE 값을 늘려주세요.")

sample_df = business_df.sample(n=300, random_state=42).reset_index(drop=True)

print("Final sampled emails:", len(sample_df))
print(sample_df[["subject", "normalized_subject"]].head(10))

# ==========================================
# 11. CSV 저장
# ==========================================
output_file = os.path.join(
    REF_PATH,
    "enron_business_email_sample_300_cleaned.csv"
)

sample_df[["subject", "body", "email_text"]].to_csv(output_file, index=False)

print("Saved ->", output_file)
sample_df[["subject", "body", "email_text"]].to_csv(output_file, index=False)

print(f"Saved -> {output_file}")

# ==========================================
# 12. 참고용 통계 출력
# ==========================================
print("\nTop sample subjects:")
print(sample_df["subject"].head(20).to_list())

```
실행 결과 해석 (정상 여부)

Total email files found: 517424

→ Enron dataset 전체 이메일 개수 정상

Parsed emails: 49526

→ 앞에서 설정한 MAX_PARSE = 50000 중
파싱 가능한 이메일 약 4.9만 개 추출

Business-like emails: 37241

→ 업무 관련 키워드 필터링 후 3.7만 개

After deduplication: 19499

→ 중복 제거 후 약 2만 개

Final sampled emails: 300

→ 그 중 300개 랜덤 샘플링
```



> Enron Email Dataset에는 약 51만 개의 이메일이 포함되어 있다.
본 연구에서는 데이터 처리 효율성을 위해 초기 5만 개 이메일을 대상으로 파싱을 수행하였다.

> 이후 업무 관련 키워드 기반 필터링을 적용하여 약 3.7만 개의 업무성 이메일을 선별하였다.
중복 이메일을 제거한 후 최종적으로 약 1.9만 개의 이메일을 확보하였으며,
이 중 300개를 무작위 샘플링하여 이메일 구조 패턴 분석에 활용하였다.



# **Ⅱ. 패턴 분석**

- subject 상위 패턴

- greeting 패턴

- closing 패턴

- body 표현 패턴

- 나중에 intent에 연결하기 쉬운 업무 행위 표현 추출

***즉,나중에 “견적 요청”, “미팅 일정 조율”, “자료 요청”, “기술 지원 요청” 같은 intent로 옮겨가기 쉽게 만드는 코드***

**2-1. CSV 불러오기 + 기본 전처리**

In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/Capstone_AI2"
REF_PATH = os.path.join(BASE_PATH, "reference_data")

file_path = os.path.join(
    REF_PATH,
    "enron_business_email_sample_300_cleaned.csv"
)

df = pd.read_csv(file_path)

print(df.shape)
df.head()

**2-2. subject 패턴 분석 코드**

In [ ]:
import re
from collections import Counter

def clean_subject_for_pattern(subject):
    if pd.isna(subject):
        return ""
    s = str(subject).strip().lower()

    # re/fw/fwd 제거
    s = re.sub(r"^(re|fw|fwd)\s*:\s*", "", s)

    # 숫자/날짜/특수문자 정리
    s = re.sub(r"\d{1,4}", "<NUM>", s)
    s = re.sub(r"[^a-zA-Z0-9\s<>/&-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    return s

df["subject_clean"] = df["subject"].apply(clean_subject_for_pattern)

subject_counter = Counter(df["subject_clean"])
top_subjects = subject_counter.most_common(50)

print("=== Top Subject Patterns ===")
for s, c in top_subjects[:30]:
    print(f"{c:3d} | {s}")

subject_df = pd.DataFrame(top_subjects, columns=["pattern", "count"])

subject_file = os.path.join(
    REF_PATH,
    "enron_subject_patterns.csv"
)

subject_df.to_csv(subject_file, index=False)

print("Saved ->", subject_file)

**2-3. greeting 패턴 분석 코드**

In [ ]:
greeting_patterns = [
    r"^(hi\b.*)",
    r"^(hello\b.*)",
    r"^(dear\b.*)",
    r"^(good morning\b.*)",
    r"^(good afternoon\b.*)",
    r"^(good evening\b.*)"
]

def extract_greeting(body):
    if pd.isna(body):
        return None

    lines = str(body).split("\n")
    for line in lines[:5]:
        line = line.strip()
        if not line:
            continue
        for p in greeting_patterns:
            m = re.match(p, line, flags=re.IGNORECASE)
            if m:
                return m.group(1).strip()
    return None

df["greeting"] = df["body"].apply(extract_greeting)

greeting_counter = Counter(df["greeting"].dropna())
print("=== Greeting Patterns ===")
for g, c in greeting_counter.most_common(30):
    print(f"{c:3d} | {g}")

greeting_df = pd.DataFrame(
    greeting_counter.most_common(),
    columns=["greeting_pattern", "count"]
)

greeting_file = os.path.join(
    REF_PATH,
    "enron_greeting_patterns.csv"
)

greeting_df.to_csv(greeting_file, index=False)

print("Saved ->", greeting_file)

**2-4. closing 패턴 분석 코드**

In [ ]:
closing_keywords = [
    "best regards",
    "regards",
    "thanks",
    "thank you",
    "sincerely",
    "best",
    "kind regards"
]

def extract_closing(body):
    if pd.isna(body):
        return None

    text = str(body).lower()
    lines = [x.strip() for x in text.split("\n") if x.strip()]
    tail = lines[-8:] if len(lines) >= 8 else lines

    for line in tail:
        for kw in closing_keywords:
            if kw in line:
                return line
    return None

df["closing"] = df["body"].apply(extract_closing)

closing_counter = Counter(df["closing"].dropna())
print("=== Closing Patterns ===")
for c_text, c in closing_counter.most_common(30):
    print(f"{c:3d} | {c_text}")

closing_df = pd.DataFrame(
    closing_counter.most_common(),
    columns=["closing_pattern", "count"]
)

closing_file = os.path.join(
    REF_PATH,
    "enron_closing_patterns.csv"
)

closing_df.to_csv(closing_file, index=False)

print("Saved ->", closing_file)

**2-5. body 요청 패턴 분석 코드**



> intent 설계에 직접 도움이 되는 업무형 요청 표현 체크 코드



In [ ]:
request_patterns = [
    r"please let me know",
    r"please review",
    r"please confirm",
    r"please find attached",
    r"attached is",
    r"attached are",
    r"i would like to",
    r"we would like to",
    r"could you",
    r"can you",
    r"request for",
    r"thank you for your",
    r"for your review",
    r"for your approval",
    r"schedule a meeting",
    r"confirm receipt",
    r"let me know your availability",
    r"please advise",
    r"please send",
    r"please provide"
]

def extract_request_patterns(text):
    if pd.isna(text):
        return []

    t = str(text).lower()
    found = []

    for p in request_patterns:
        if re.search(p, t):
            found.append(p)

    return found

df["matched_patterns"] = df["email_text"].apply(extract_request_patterns)

pattern_counter = Counter()
for items in df["matched_patterns"]:
    for item in items:
        pattern_counter[item] += 1

print("=== Business Request / Action Patterns ===")
for p, c in pattern_counter.most_common():
    print(f"{c:3d} | {p}")

request_df = pd.DataFrame(
    pattern_counter.most_common(),
    columns=["request_pattern", "count"]
)

request_file = os.path.join(
    REF_PATH,
    "enron_request_patterns.csv"
)

request_df.to_csv(request_file, index=False)

print("Saved ->", request_file)

**2-6. intent 연결용 패턴 분석 테이블 생성 코드**

In [ ]:
def infer_pattern_type(text):
    t = str(text).lower()

    if re.search(r"quote|quotation|pricing|price|cost", t):
        return "견적/가격 문의"
    elif re.search(r"contract|agreement|proposal", t):
        return "계약/제안 관련"
    elif re.search(r"meeting|schedule|availability|conference|call", t):
        return "일정 조율"
    elif re.search(r"invoice|payment|budget", t):
        return "재무/정산 관련"
    elif re.search(r"support|issue|problem|system|access|account", t):
        return "기술 지원/시스템 관련"
    elif re.search(r"report|update|announcement|notice", t):
        return "보고/공지"
    elif re.search(r"interview|candidate|application|resume|leave|vacation", t):
        return "인사/채용 관련"
    elif re.search(r"document|attached|review|approval|provide|send", t):
        return "자료 요청/검토 요청"
    else:
        return "기타 업무 패턴"

df["pattern_type"] = df["email_text"].apply(infer_pattern_type)

pattern_summary = df.groupby("pattern_type").agg(
    count=("pattern_type", "count"),
    sample_subject=("subject", "first")
).reset_index().sort_values(by="count", ascending=False)

pattern_summary

pattern_file = os.path.join(
    REF_PATH,
    "enron_pattern_summary.csv"
)

pattern_summary.to_csv(pattern_file, index=False)

print("Saved ->", pattern_file)

# **Ⅲ. Enron 패턴 → 30개 intent 자동 매핑**

In [ ]:
import os
import pandas as pd

# ==========================================
# 0. 저장 경로
# ==========================================
BASE_PATH = "/content/drive/MyDrive/Capstone_AI2"
REF_PATH = os.path.join(BASE_PATH, "reference_data")
os.makedirs(REF_PATH, exist_ok=True)

# ==========================================
# 1. Domain / Intent 설계 테이블 정의 (총 30개)
# ==========================================
intent_design_rows = [
    # ---------------- Sales ----------------
    ["Sales", "견적 요청", "가격 문의 및 견적 요청 표현", "Request for quotation / Price inquiry", "Greeting → Request → Detail → Closing"],
    ["Sales", "가격 협상", "가격 조정 및 조건 협의 표현", "Pricing discussion / Rate adjustment", "Greeting → Negotiation → Condition → Closing"],
    ["Sales", "계약 문의", "계약 조건 및 합의 사항 확인 표현", "Contract inquiry / Agreement details", "Greeting → Inquiry → Clarification → Closing"],
    ["Sales", "미팅 일정 조율", "일정 조율 및 회의 협의 표현", "Meeting schedule / Availability", "Greeting → Scheduling → Confirmation → Closing"],
    ["Sales", "제안 요청", "제안서 제출 요청 표현", "Request for proposal / Proposal request", "Greeting → Request → Deadline → Closing"],

    # ------------ Marketing & PR ------------
    ["Marketing & PR", "광고 문의", "광고/마케팅 문의 표현", "Advertising inquiry / Marketing inquiry", "Greeting → Inquiry → Campaign detail → Closing"],
    ["Marketing & PR", "보도자료 요청", "보도자료 및 미디어 자료 요청 표현", "Press release request / Media inquiry", "Greeting → Request → Media context → Closing"],
    ["Marketing & PR", "인터뷰 요청", "인터뷰 섭외 및 질의 요청 표현", "Interview request / Media interview", "Greeting → Request → Topic → Closing"],
    ["Marketing & PR", "협찬/제휴 제안", "파트너십 및 협찬 제안 표현", "Sponsorship / Partnership proposal", "Greeting → Proposal → Benefit → Closing"],
    ["Marketing & PR", "콘텐츠 협업 문의", "콘텐츠 공동 작업 및 협업 문의 표현", "Content collaboration inquiry", "Greeting → Proposal → Collaboration detail → Closing"],
    ["Marketing & PR", "행사/캠페인 문의", "행사 또는 캠페인 관련 문의 표현", "Event / Campaign inquiry", "Greeting → Inquiry → Event detail → Closing"],

    # ---------------- HR ----------------
    ["HR", "채용 문의", "채용 및 포지션 문의 표현", "Job inquiry / Position inquiry", "Greeting → Inquiry → Background → Closing"],
    ["HR", "면접 조율", "면접 일정 협의 표현", "Interview scheduling", "Greeting → Schedule → Availability → Closing"],
    ["HR", "휴가 신청", "휴가 및 부재 승인 요청 표현", "Leave request / Vacation request", "Greeting → Request → Date → Closing"],
    ["HR", "증명서 발급 요청", "증명 문서 요청 표현", "Certificate request / Employment verification", "Greeting → Request → Purpose → Closing"],

    # ---------------- Finance ----------------
    ["Finance", "세금계산서 요청", "청구서/세금계산서 요청 표현", "Invoice request / Billing request", "Greeting → Request → Transaction detail → Closing"],
    ["Finance", "비용 처리 문의", "비용 처리 및 지출 관련 문의 표현", "Expense processing inquiry / Reimbursement inquiry", "Greeting → Inquiry → Expense detail → Closing"],
    ["Finance", "입금 확인 요청", "결제 및 입금 확인 표현", "Payment confirmation / Receipt confirmation", "Greeting → Confirmation request → Detail → Closing"],
    ["Finance", "정산 문의", "비용 및 정산 확인 표현", "Settlement inquiry / Reconciliation inquiry", "Greeting → Inquiry → Amount detail → Closing"],

    # ------------ Customer Support ------------
    ["Customer Support", "불만 접수", "문제 제기 및 불만 전달 표현", "Complaint / Service issue", "Greeting → Complaint → Detail → Closing"],
    ["Customer Support", "기술 지원 요청", "시스템 문제 및 기술 지원 표현", "Technical support request / Help request", "Greeting → Problem → Request → Closing"],
    ["Customer Support", "환불 요청", "환불 및 반환 요청 표현", "Refund request", "Greeting → Issue → Refund request → Closing"],
    ["Customer Support", "사용법 문의", "서비스/시스템 사용 문의 표현", "Usage inquiry / How to use", "Greeting → Question → Context → Closing"],

    # ---------------- IT/Ops ----------------
    ["IT/Ops", "시스템 오류 보고", "오류 및 장애 보고 표현", "Error report / Failure notice", "Greeting → Error description → Request → Closing"],
    ["IT/Ops", "계정 생성 요청", "신규 계정 생성 요청 표현", "Account creation request / New account setup", "Greeting → Request → User info → Closing"],
    ["IT/Ops", "접근 권한 변경", "접근 권한 조정 요청 표현", "Access change request / Permission update", "Greeting → Request → Permission detail → Closing"],

    # ---------------- Admin ----------------
    ["Admin", "공지 전달", "전체 공지 및 전달 표현", "Announcement / Notice", "Greeting → Announcement → Detail → Closing"],
    ["Admin", "내부 보고", "업무 상태 및 결과 보고 표현", "Weekly report / Status update", "Greeting → Report → Summary → Closing"],
    ["Admin", "자료 요청", "문서 및 파일 요청 표현", "Document request / File request", "Greeting → Request → Needed item → Closing"],
    ["Admin", "협조 요청", "업무 협조 및 지원 요청 표현", "Request for cooperation / Support request", "Greeting → Request → Reason → Closing"],
]

intent_design_df = pd.DataFrame(
    intent_design_rows,
    columns=["Domain", "Intent", "Email Pattern", "Subject Pattern", "Example Structure"]
)

print("총 intent 개수:", len(intent_design_df))
display(intent_design_df)

# ==========================================
# 2. CSV 저장
# ==========================================
csv_file = os.path.join(REF_PATH, "intent_design_table_30.csv")
intent_design_df.to_csv(csv_file, index=False, encoding="utf-8-sig")

print("Saved ->", csv_file)